# 카메라 기반 일별 활력도 변화와 향후 3일 이내 돼지 질병 확인 위험의 연관성 평가

**논문/학위논문 초고용 Jupyter Notebook 초안**

이 노트북은 `docs/01_experiment_plan.md`의 연구설계와 `data/raw/pig_daily_observations.csv` 원자료를 기준으로,
Methods → Results → Discussion 문장을 바로 옮길 수 있게 구성한 분석 초안입니다.

| 구분 | 내용 |
|---|---|
| 설계 | 전향적 반복측정 관찰 코호트(합성 예시) |
| 대상 | 8개 돈방, 돼지 120두, 42일 |
| 측정단위 | 개체-일 (pig-day) |
| 결과변수 | `disease_next_3d` (다음날~3일 이내 임상발현) |
| 주지표 | AUPRC |

> **재현:** 제출·공유 전 Kernel → **Restart Kernel and Run All Cells**로 처음부터 실행하세요.  
> **한계:** 본 자료는 실제 농장과 무관한 **합성데이터**이며, 활력도 저하만으로 특정 질병을 진단하거나 인과를 주장하지 않습니다.


## 0. Notebook 사용 원칙

| 셀 | 역할 |
|---|---|
| **Markdown** | 연구질문·가설·제외기준·해석·논문 문장 초고 |
| **Code** | 데이터 처리·통계·시각화 (작은 단계로 실행) |

- 판단 기준(주지표, 분할, 임계값)은 **결과를 보기 전에** Markdown에 고정합니다.
- 제외 기준을 중간에 바꾸면 변경 시점·이유·영향을 이 노트북에 남깁니다.
- 모델 출력은 진단이 아니라 **관찰 우선순위를 정하는 보조정보**입니다.


## 1. 연구 배경 (Introduction 초고)

돼지는 질병 초기에 이동량, 섭식, 음수, 자세 변화와 사회적 접촉이 달라질 수 있다.
사람의 순회 관찰만으로 미세한 변화를 연속적으로 확인하기 어렵기 때문에,
영상과 환경센서에서 얻은 행동지표를 이용해 수의사 확인이 필요한 개체를 조기에 선별하는 방법을 검토한다.

본 분석은 활력도 지표의 가능성을 평가하는 **관찰연구 예시**다.
합성데이터의 `vitality_score`는 활동시간·이동거리·섭식시간 증가 시 상승하고,
누움 비율·기침 횟수 증가 시 하락하도록 조합한 교육용 점수(0–100)이며,
임상적으로 검증된 지표가 아니다.


## 2. 연구 질문 · 가설 · 사전 판단 기준

### 연구 질문
1. 질병 확인 1~3일 전 개체의 활력도는 해당 개체의 최근 기준선보다 낮아지는가?
2. 단일 시점 활력도보다 최근 3일 변화량을 포함한 모델의 예측력이 높은가?
3. 환경온도, 습도, 돈방 차이를 고려한 뒤에도 활력도 변화가 조기경보에 기여하는가?

### 가설
- **H1:** `vitality_delta_3d`가 감소할수록 `disease_next_3d=1`의 확률이 증가한다.
- **H2:** 활동량·섭식·누움·기침 변화가 결합된 모델(M2)은 기초모델(M0)보다 **AUPRC**가 높다.
- **H3:** 고온·다습 등 환경성 활력 저하를 보정하면 불필요한 경보가 감소한다.

### 사전 고정 판단 기준 (결과를 보기 전에 기록)

| 항목 | 기준 |
|---|---|
| 측정 단위 | 개체-일 (pig-day) |
| 결과변수 | `disease_next_3d`: 관찰일 **다음날부터 3일 이내** 임상발현 |
| 주지표 | **AUPRC** (양성률이 낮아 정확도는 주지표로 쓰지 않음) |
| 보조지표 | AUROC, 민감도, 특이도, PPV, 돼지 100두·일당 경보 수 |
| 데이터 분할 | train 0–25일 / validation 26–33일 / test 34–41일 (**시간순**, 임의 행분할 금지) |
| 임계값 | validation에서 목표 민감도 ≥ 0.80을 만족하는 후보 중 **정밀도 최대** |
| H2 지지 조건 | 동일 test에서 M0 → M1 → M2로 AUPRC가 개선 |

### 입력 금지 (누수 방지)
`clinical_onset_date`, `confirmation_date`, `treatment_date`, `recovery_date`, `days_to_onset` 및
당일 발병 **이후** 기록은 조기예측 분석·모델 입력에 사용하지 않는다.


## 3. 환경 설정

프로젝트 루트를 기준으로 경로를 잡고, 분석에 필요한 라이브러리를 불러온다.


In [ ]:
from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    precision_recall_curve,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 40)
sns.set_theme(style="whitegrid", context="notebook")

# 한글 figure 라벨 폰트
# 원인: Apple SD Gothic Neo(.ttc)는 matplotlib이 이름을 찾아도 한글 글리프를
#       못 그려 □(tofu)가 나는 경우가 많다. 단일 TTF를 직접 등록하고,
#       seaborn.set_theme 이후에 다시 고정하는 것이 안전하다.
import matplotlib.font_manager as fm

def configure_korean_font():
    """한글 가능 TTF를 matplotlib에 등록하고 rcParams에 고정한다."""
    font_files = [
        Path("/System/Library/Fonts/Supplemental/AppleGothic.ttf"),  # macOS TTF (권장)
        Path("/Library/Fonts/AppleGothic.ttf"),
        Path("/System/Library/Fonts/AppleSDGothicNeo.ttc"),  # fallback (TTC, 불안정할 수 있음)
        Path.home() / "Library/Fonts/NanumGothic.ttf",
        Path("/usr/share/fonts/truetype/nanum/NanumGothic.ttf"),
    ]

    chosen_path = next((p for p in font_files if p.exists()), None)
    if chosen_path is None:
        print("경고: 한글 폰트 파일을 찾지 못했습니다.")
        print("  macOS: AppleGothic.ttf 또는 NanumGothic 설치 후 커널 재시작")
        return None

    # 캐시된 잘못된 매핑을 피하기 위해 폰트 매니저를 갱신한 뒤 파일을 등록
    try:
        fm.fontManager.addfont(str(chosen_path))
    except Exception as exc:
        print("addfont 실패:", exc)

    prop = fm.FontProperties(fname=str(chosen_path))
    name = prop.get_name()

    # DejaVu 등으로 폴백되면 한글이 다시 깨지므로 해당 폰트만 사용
    plt.rcParams["font.family"] = name
    plt.rcParams["font.sans-serif"] = [name]
    plt.rcParams["axes.unicode_minus"] = False
    print(f"한글 폰트: {name} ({chosen_path})")
    return prop

def apply_korean_font(fig, fontprop=None):
    """이미 그려진 figure의 모든 텍스트에 한글 폰트를 강제 적용."""
    prop = fontprop or KOREAN_FONT
    if prop is None:
        return
    for ax in fig.axes:
        for item in [ax.title, ax.xaxis.label, ax.yaxis.label]:
            item.set_fontproperties(prop)
        for label in ax.get_xticklabels() + ax.get_yticklabels():
            label.set_fontproperties(prop)
        legend = ax.get_legend()
        if legend is not None:
            for text in legend.get_texts():
                text.set_fontproperties(prop)
            if legend.get_title() is not None:
                legend.get_title().set_fontproperties(prop)

KOREAN_FONT = configure_korean_font()

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
elif not (ROOT / "data" / "raw" / "pig_daily_observations.csv").exists():
    candidate = Path(".").resolve()
    ROOT = candidate if (candidate / "data").exists() else Path.cwd()

RAW = ROOT / "data" / "raw"
CLEAN = ROOT / "data" / "clean"
RESULTS = ROOT / "results"
RESULTS.mkdir(exist_ok=True)

print("ROOT:", ROOT)
print("원자료:", (RAW / "pig_daily_observations.csv").exists())
print("정제자료:", (CLEAN / "practice_dataset.csv").exists())


## 4. 자료 불러오기

| 파일 | 역할 |
|---|---|
| `pig_daily_observations.csv` | 개체-일 원 관찰자료 (중복·결측·범위오류 포함) |
| `pig_metadata.csv` | 개체당 1행 메타데이터 |
| `disease_events.csv` | 수의사 확인 질병 이벤트 |
| `practice_dataset.csv` | QC·라벨·과거 변화량이 반영된 분석용 자료 |

조인 키는 `pig_id`, 반복측정 키는 `(pig_id, observation_date)`다.


In [ ]:
observations = pd.read_csv(
    RAW / "pig_daily_observations.csv",
    parse_dates=["observation_date"],
)
metadata = pd.read_csv(
    RAW / "pig_metadata.csv",
    parse_dates=["birth_date", "arrival_date"],
)
events = pd.read_csv(
    RAW / "disease_events.csv",
    parse_dates=[
        "clinical_onset_date",
        "confirmation_date",
        "treatment_date",
        "recovery_date",
    ],
)
practice = pd.read_csv(
    CLEAN / "practice_dataset.csv",
    parse_dates=["observation_date"],
)

summary = pd.DataFrame({
    "file": [
        "pig_daily_observations",
        "pig_metadata",
        "disease_events",
        "practice_dataset",
    ],
    "n_rows": [len(observations), len(metadata), len(events), len(practice)],
    "n_cols": [
        observations.shape[1],
        metadata.shape[1],
        events.shape[1],
        practice.shape[1],
    ],
    "n_pigs": [
        observations["pig_id"].nunique(),
        metadata["pig_id"].nunique(),
        events["pig_id"].nunique(),
        practice["pig_id"].nunique(),
    ],
    "date_min": [
        observations["observation_date"].min(),
        metadata["arrival_date"].min(),
        events["clinical_onset_date"].min(),
        practice["observation_date"].min(),
    ],
    "date_max": [
        observations["observation_date"].max(),
        metadata["arrival_date"].max(),
        events["clinical_onset_date"].max(),
        practice["observation_date"].max(),
    ],
})
summary


In [ ]:
print("=== pig_daily_observations 열 ===")
print(list(observations.columns))
print("\n=== 앞 5행 ===")
display(observations.head())
print("\n=== 수치형 요약 ===")
display(observations.describe(include="number").T.round(2))


### Methods용 해석 메모 (작성란)
- 원 관찰자료 규모: ____ 행, 개체 ____ 두, 기간 ____ ~ ____
- 메타데이터: 개체당 1행 (____ 두)
- 질병 이벤트: ____ 건
- `practice_dataset`은 QC·라벨·과거 변화량이 반영된 **분석 대상 자료**로 사용한다.


## 5. 데이터 품질 점검 (실험계획서 §8)

사전 제외·품질 기준 (결과를 본 뒤 유리하게 변경하지 않음):

1. `(pig_id, observation_date)` 완전 중복 제거
2. `valid=False` 제외 후 사유(`qc_note`) 집계
3. 카메라 커버리지 70% 미만 제외
4. 이동거리 음수, 누움 비율 0–1 밖, 커버리지 0–100 밖 제외
5. 핵심 영상변수 결측 행 제외 또는 사전 정의된 대치
6. 메타데이터 병합은 `many_to_one` 검증
7. 원자료와 정제자료를 별도 보존하고 제외 건수를 기록


In [ ]:
qc_report = {}

# 1) 중복
n_dup = int(observations.duplicated(["pig_id", "observation_date"]).sum())
qc_report["duplicate_pig_days"] = n_dup

# 2) valid / qc_note
qc_report["n_invalid"] = int((~observations["valid"].astype(bool)).sum())
print("=== valid 분포 ===")
print(observations["valid"].value_counts(dropna=False))
print("\n=== qc_note Top ===")
print(observations["qc_note"].fillna("(없음)").value_counts().head(10))

# 3–4) 범위·커버리지
qc_report["coverage_lt_70"] = int((observations["camera_coverage_pct"] < 70).sum())
qc_report["distance_negative"] = int((observations["distance_m"] < 0).sum())
qc_report["lying_out_of_range"] = int(
    ((observations["lying_ratio"] < 0) | (observations["lying_ratio"] > 1)).sum()
)
qc_report["coverage_out_of_range"] = int(
    (
        (observations["camera_coverage_pct"] < 0)
        | (observations["camera_coverage_pct"] > 100)
    ).sum()
)

# 5) 결측
missing = observations.isna().sum().sort_values(ascending=False)
print("\n=== 결측 Top 10 ===")
print(missing.head(10))
qc_report["rows_any_missing"] = int(observations.isna().any(axis=1).sum())

# 6) 메타데이터 매칭
obs_ids = set(observations["pig_id"])
meta_ids = set(metadata["pig_id"])
qc_report["obs_not_in_meta"] = len(obs_ids - meta_ids)
qc_report["meta_not_in_obs"] = len(meta_ids - obs_ids)

print("\n=== 품질 요약 ===")
pd.Series(qc_report)


In [ ]:
# 제외 기준을 단계적으로 적용해 건수를 기록 (원자료는 수정하지 않음)
df = observations.copy()
exclusion_log = []

def log_exclude(name, before, after):
    exclusion_log.append({"step": name, "before": before, "removed": before - after, "after": after})

before = len(df)
df = df.drop_duplicates(["pig_id", "observation_date"], keep="first")
log_exclude("drop_duplicate_pig_days", before, len(df))

before = len(df)
df = df.loc[df["valid"].astype(bool)].copy()
log_exclude("valid_True_only", before, len(df))

before = len(df)
df = df.loc[df["camera_coverage_pct"] >= 70].copy()
log_exclude("coverage_ge_70", before, len(df))

before = len(df)
ok_distance = df["distance_m"].isna() | (df["distance_m"] >= 0)
ok_lying = df["lying_ratio"].isna() | df["lying_ratio"].between(0, 1)
ok_cov = df["camera_coverage_pct"].between(0, 100)
df = df.loc[ok_distance & ok_lying & ok_cov].copy()
log_exclude("range_checks", before, len(df))

core_cols = [
    "activity_minutes", "distance_m", "lying_ratio",
    "feeding_minutes", "vitality_score", "ambient_temp_c", "humidity_pct",
]
before = len(df)
df = df.dropna(subset=core_cols).copy()
log_exclude("drop_core_missing", before, len(df))

exclusion_table = pd.DataFrame(exclusion_log)
print(f"원자료 {len(observations)}행 → QC 후 {len(df)}행")
exclusion_table


### 품질 보고 (Methods에 넣을 4문장 초고)

1. 원 관찰자료 ____행에서 `(pig_id, observation_date)` 중복 ____건, `valid=False` ____건, 커버리지 70% 미만 ____건을 점검했다.
2. 범위 오류(이동거리 음수, 누움 비율·커버리지 범위 밖) ____건을 제외했다.
3. 핵심 영상·환경변수 결측 행을 제외한 뒤 분석 후보 ____행이 남았다.
4. 이후 모델링은 동일 기준이 반영된 `practice_dataset`을 사용하며, 원자료와 정제자료를 분리 보존했다.


## 6. 라벨 정의와 분석 대상

- **예측시점:** 매일 관찰 종료 시점
- **예측범위:** 다음날부터 3일 이내
- **라벨:** `disease_next_3d=1`이면 해당 기간에 `clinical_onset_date`가 존재
- **분할:** `study_day` 기준 train / validation / test (시간순)

정제자료의 `vitality_delta_3d`는 **관찰일 이전** 최대 3일 활력도 중앙값 대비 변화량이다
(현재·미래값 미사용 → 누수 방지).


In [ ]:
label_by_split = (
    practice.groupby("split", sort=False)["disease_next_3d"]
    .agg(n="size", positives="sum", prevalence="mean")
    .reindex(["train", "validation", "test"])
)
label_by_split["prevalence_pct"] = (100 * label_by_split["prevalence"]).round(2)
print("분할별 양성률")
display(label_by_split)

print("\n질병 이벤트 증후군 × 중증도")
display(events.groupby(["syndrome", "severity"]).size().unstack(fill_value=0))

print("\n양성 vs 음성 개체-일 요약")
display(
    practice.groupby("disease_next_3d")[
        ["vitality_score", "vitality_delta_3d", "activity_minutes", "cough_events"]
    ]
    .agg(["mean", "median", "std"])
    .round(2)
)


### 해석 (작성란)
- 양성률이 낮으므로 **정확도**보다 **AUPRC**와 경보량(alerts/100 pig-days)을 함께 본다.
- 양성 군에서 `vitality_delta_3d` 평균이 더 낮으면 H1을 기술적으로 지지하는 근거가 된다.
- 메모: _______________________________________________


## 7. 기술통계 · 탐색적 시각화 (Results용 그림)

실험계획서 §9.1에 따라 다음을 확인한다.

- 양성·음성 개체-일의 활력도 변화량 분포
- 온도와 활력도의 관계 (환경 교란)
- 임상발현 전 7일간 지표 궤적
- 돈방별 관찰 수·양성률


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")

sns.boxplot(
    data=practice, x="disease_next_3d", y="vitality_delta_3d",
    ax=axes[0], color="#DCEFEB",
)
sns.stripplot(
    data=practice.sample(n=min(800, len(practice)), random_state=42),
    x="disease_next_3d", y="vitality_delta_3d",
    ax=axes[0], color="0.25", alpha=0.2, size=2,
)
axes[0].axhline(0, color="0.5", ls="--", lw=1)
axes[0].set(
    xlabel="disease_next_3d",
    ylabel="vitality_delta_3d (현재 - 이전 3일 중앙값)",
    title="라벨별 활력도 3일 변화량",
)

sns.scatterplot(
    data=practice.sample(n=min(1200, len(practice)), random_state=0),
    x="ambient_temp_c", y="vitality_score",
    hue="disease_next_3d", alpha=0.35, s=18, ax=axes[1],
)
axes[1].set(xlabel="ambient_temp_c (°C)", ylabel="vitality_score", title="온도와 활력도")
axes[1].legend(title="disease_next_3d", loc="best")

apply_korean_font(fig)
fig.savefig(RESULTS / "thesis_eda_vitality.png", dpi=160)
plt.show()
print("저장:", RESULTS / "thesis_eda_vitality.png")


In [ ]:
# 임상발현 전 7일 궤적 (질병 이벤트 기준, 과거 관찰만 정렬)
onset = events[["pig_id", "clinical_onset_date", "syndrome"]].copy()
traj = practice.merge(onset, on="pig_id", how="inner")
traj["days_to_onset"] = (
    traj["clinical_onset_date"] - traj["observation_date"]
).dt.days
traj = traj.loc[traj["days_to_onset"].between(1, 7)].copy()

daily = (
    traj.groupby("days_to_onset")[["vitality_score", "vitality_delta_3d", "activity_minutes"]]
    .mean()
    .sort_index(ascending=False)
)

fig, ax = plt.subplots(figsize=(7, 4), layout="constrained")
ax.plot(daily.index, daily["vitality_score"], marker="o", label="vitality_score")
ax.plot(daily.index, daily["vitality_delta_3d"], marker="s", label="vitality_delta_3d")
ax.invert_xaxis()
ax.set(
    xlabel="Days before clinical onset",
    ylabel="Mean value",
    title="임상발현 전 7일 평균 궤적",
)
ax.legend()
apply_korean_font(fig)
fig.savefig(RESULTS / "thesis_onset_trajectory.png", dpi=160)
plt.show()
display(daily.round(2))
print("저장:", RESULTS / "thesis_onset_trajectory.png")


In [ ]:
pen_summary = (
    practice.groupby("pen_id")
    .agg(
        n_rows=("pig_id", "size"),
        n_pigs=("pig_id", "nunique"),
        mean_vitality=("vitality_score", "mean"),
        mean_delta=("vitality_delta_3d", "mean"),
        positive_rate=("disease_next_3d", "mean"),
        mean_temp=("ambient_temp_c", "mean"),
        mean_coverage=("camera_coverage_pct", "mean"),
    )
    .round(3)
)
pen_summary


## 8. 비교 모델 정의 (실험계획서 §9.2)

해석 가능한 **로지스틱 회귀**를 기본으로 한다. 복잡한 모델은 M2 이후 확장 과제로만 사용한다.

| 모델 | 포함 변수 |
|---|---|
| **M0** 기초 | 나이, 체중, 온도, 습도, 돈방 |
| **M1** 현재 활력 | M0 + 당일 행동·활력도·카메라 커버리지 |
| **M2** 변화 | M1 + 최근 3일 기준선 대비 변화량 (`vitality_delta_3d` 등) |


In [ ]:
TARGET = "disease_next_3d"

FEATURE_SETS = {
    "M0_baseline": [
        "age_days", "weight_kg", "ambient_temp_c", "humidity_pct", "pen_id",
    ],
    "M1_current_vitality": [
        "age_days", "weight_kg", "ambient_temp_c", "humidity_pct", "pen_id",
        "activity_minutes", "distance_m", "lying_ratio", "feeding_visits",
        "feeding_minutes", "drinking_visits", "social_contacts",
        "posture_changes", "cough_events", "vitality_score",
        "camera_coverage_pct",
    ],
    "M2_change": [
        "age_days", "weight_kg", "ambient_temp_c", "humidity_pct", "pen_id",
        "activity_minutes", "distance_m", "lying_ratio", "feeding_visits",
        "feeding_minutes", "drinking_visits", "social_contacts",
        "posture_changes", "cough_events", "vitality_score",
        "camera_coverage_pct",
        "vitality_delta_3d", "activity_delta_pct_3d", "cough_events_prior_3d",
    ],
}


def build_model(features):
    categorical = [c for c in features if c in {"pen_id", "sex", "breed"}]
    numeric = [c for c in features if c not in categorical]
    prep = ColumnTransformer([
        (
            "num",
            Pipeline([
                ("impute", SimpleImputer(strategy="median")),
                ("scale", StandardScaler()),
            ]),
            numeric,
        ),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
    ])
    return Pipeline([
        ("prep", prep),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced")),
    ])


def choose_threshold(y_true, probabilities, min_sensitivity=0.80):
    """validation에서 민감도 ≥ 0.80을 만족하는 임계값 중 정밀도 최대."""
    precision, recall, thresholds = precision_recall_curve(y_true, probabilities)
    # precision/recall length = len(thresholds)+1; align to thresholds
    precision, recall = precision[:-1], recall[:-1]
    if len(thresholds) == 0:
        return 0.5
    eligible = recall >= min_sensitivity
    if not np.any(eligible):
        # 목표 민감도를 못 맞추면 민감도 최대 지점
        return float(thresholds[int(np.argmax(recall))])
    idx = np.where(eligible)[0]
    best_local = idx[int(np.argmax(precision[eligible]))]
    return float(thresholds[best_local])


def compute_metrics(y_true, probabilities, threshold):
    pred = probabilities >= threshold
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        "n": int(len(y_true)),
        "positives": int(np.sum(y_true)),
        "auprc": float(average_precision_score(y_true, probabilities)),
        "auroc": float(roc_auc_score(y_true, probabilities)),
        "threshold": float(threshold),
        "sensitivity": float(tp / (tp + fn) if tp + fn else 0),
        "specificity": float(tn / (tn + fp) if tn + fp else 0),
        "ppv": float(tp / (tp + fp) if tp + fp else 0),
        "alerts_per_100_pig_days": float(100 * (tp + fp) / len(y_true)),
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
    }


print("모델별 특성 수:", {k: len(v) for k, v in FEATURE_SETS.items()})


## 9. 학습 · 검증 · 최종 평가 (실험계획서 §9.3–9.4)

1. **train**으로 적합  
2. **validation**에서 임계값 선택 (민감도 ≥ 0.80)  
3. **test는 최종 1회만** 평가  

동일 개체의 인접 날짜를 무작위로 행 분할하지 않는다.


In [ ]:
needed = sorted(set().union(*FEATURE_SETS.values()))
model_df = practice.dropna(subset=needed + [TARGET]).copy()

train = model_df.loc[model_df["split"].eq("train")]
validation = model_df.loc[model_df["split"].eq("validation")]
test = model_df.loc[model_df["split"].eq("test")]

print("분석 행 수:", len(model_df))
print("train / val / test:", len(train), len(validation), len(test))
print("test 양성률:", round(test[TARGET].mean(), 4))

all_results = {}
prediction_frame = test[["observation_date", "pig_id", "pen_id", TARGET]].copy()

for name, features in FEATURE_SETS.items():
    model = build_model(features)
    model.fit(train[features], train[TARGET])
    val_prob = model.predict_proba(validation[features])[:, 1]
    threshold = choose_threshold(validation[TARGET].to_numpy(), val_prob)
    test_prob = model.predict_proba(test[features])[:, 1]
    all_results[name] = compute_metrics(test[TARGET].to_numpy(), test_prob, threshold)
    prediction_frame[f"prob_{name}"] = test_prob

metric_table = pd.DataFrame(all_results).T.reset_index(names="model")
cols = [
    "model", "auprc", "auroc", "sensitivity", "specificity", "ppv",
    "alerts_per_100_pig_days", "threshold",
]
metric_table[cols].round(3)


In [ ]:
(RESULTS / "thesis_metrics.json").write_text(
    json.dumps(all_results, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
metric_table.to_csv(RESULTS / "thesis_model_comparison.csv", index=False)
prediction_frame.to_csv(RESULTS / "thesis_test_predictions.csv", index=False)
exclusion_table.to_csv(RESULTS / "thesis_exclusion_log.csv", index=False)

best_prob = prediction_frame["prob_M2_change"]
precision, recall, _ = precision_recall_curve(test[TARGET], best_prob)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
axes[0].plot(recall, precision, color="#00A99D", linewidth=2)
axes[0].axhline(test[TARGET].mean(), color="0.5", linestyle="--", label="prevalence")
axes[0].set(xlabel="Recall", ylabel="Precision", title="Test Precision–Recall (M2)")
axes[0].legend()

sns.boxplot(
    data=prediction_frame, x=TARGET, y="prob_M2_change",
    ax=axes[1], color="#DCEFEB",
)
sns.stripplot(
    data=prediction_frame, x=TARGET, y="prob_M2_change",
    ax=axes[1], color="0.2", alpha=0.25, size=3,
)
axes[1].set(
    xlabel="Disease within next 3 days",
    ylabel="Predicted probability (M2)",
    title="Test risk distribution",
)

apply_korean_font(fig)
fig.savefig(RESULTS / "thesis_model_evaluation.png", dpi=180)
plt.show()
print("저장:", RESULTS / "thesis_model_evaluation.png")


## 10. 결과 해석 (Results / Discussion 초고)

아래 문장은 **실행 결과 표를 확인한 뒤** 숫자를 채워 논문에 옮긴다.

### Results (초안)
1. 동일 test 자료에서 M0 AUPRC=____, M1=____, M2=____였다.
2. validation에서 민감도 ≥ 0.80 기준으로 선택한 임계값에서, test 민감도=____, PPV=____, 경보량=____ /100 pig-days였다.
3. 양성 개체-일의 `vitality_delta_3d` 평균은 ____로 음성(____)보다 낮았다. (H1)
4. 임상발현 전 7일 궤적에서 활력도는 발현일에 가까워질수록 ____하는 경향을 보였다.

### Discussion (편향·한계 — 실험계획서 §10)
- 고온·다습으로 인한 활동 감소가 질병처럼 보일 수 있다.
- 카메라 사각지대가 많은 개체는 활력도가 낮게 측정될 수 있다.
- 수의사 확인 시점이 실제 임상발현보다 늦을 수 있다.
- 돈방·사양관리 차이가 모델의 지름길이 될 수 있다.
- 본 자료는 합성데이터이며 외부(다른 농장) 일반화 검증이 없다.

작성란: _______________________________________________


In [ ]:
focus = metric_table.set_index("model")[
    ["auprc", "auroc", "sensitivity", "ppv", "alerts_per_100_pig_days"]
]
print("=== Test 성능 요약 ===")
display(focus.round(3))

print(
    "\nH2 체크: M2 AUPRC > M0 AUPRC ?",
    all_results["M2_change"]["auprc"] > all_results["M0_baseline"]["auprc"],
)
print(
    "H2 체크: M2 AUPRC > M1 AUPRC ?",
    all_results["M2_change"]["auprc"] > all_results["M1_current_vitality"]["auprc"],
)

# 오탐·미탐 사례를 돈방·환경 관점에서 훑기 (Discussion용)
thr = all_results["M2_change"]["threshold"]
tmp = prediction_frame.copy()
tmp["pred"] = (tmp["prob_M2_change"] >= thr).astype(int)
tmp = tmp.merge(
    test[["pig_id", "observation_date", "ambient_temp_c", "humidity_pct", "camera_coverage_pct"]],
    on=["pig_id", "observation_date"],
    how="left",
)
tmp["error_type"] = np.select(
    [
        (tmp[TARGET] == 1) & (tmp["pred"] == 0),
        (tmp[TARGET] == 0) & (tmp["pred"] == 1),
    ],
    ["FN_miss", "FP_false_alarm"],
    default="OK",
)
print("\n=== test 오류 유형 × 돈방 ===")
display(pd.crosstab(tmp["pen_id"], tmp["error_type"]))


## 11. 결론 4문장 (제출물 양식)

1. **데이터:** 8개 돈방·120두·42일 합성 관찰코호트의 개체-일 자료를 시간순으로 분할해 분석했다. QC 후 분석 표본 ____행, 질병 이벤트 ____건.
2. **결과:** 기초모델 대비 활력도·3일 변화량을 포함한 M2의 test AUPRC는 ____였고, 민감도 ____ / 경보량 ____ /100 pig-days였다.
3. **활용:** 모델 점수는 진단이 아니라, 수의사·관리자가 **먼저 관찰할 개체**를 정하는 보조정보로 사용한다.
4. **한계:** 합성데이터, 라벨 지연 가능성, 돈방·환경 교란, 외부 농장 검증 부재를 명시한다.


## 12. 재현 · 윤리 체크리스트

- [ ] Kernel **Restart & Run All**로 처음부터 오류 없이 끝까지 실행됨
- [ ] 경로가 `ROOT` 기준이라 다른 작업 디렉터리에서도 동작함
- [ ] 제외 기준을 결과 확인 후 바꾸지 않았음 (바꿨다면 시점·이유 기록)
- [ ] test는 임계값 선택에 사용하지 않았음
- [ ] 치료일·회복일·확인일을 예측변수로 쓰지 않았음
- [ ] `results/`에 metrics·그림·제외 로그가 저장됨
- [ ] AI 사용 도구·목적·입력 범위·수정 내역을 연구노트에 기록함
- [ ] 미공개 원본 영상·산학협력 자료를 공개형 AI에 입력하지 않음
